# VWAP strategies

Three strategies on the session VWAP of five exchanges' perpetuals (Binance, OKX, Coinbase International, Bybit and Deribit; the 1-minute candles, the volume of each in the base coin), traded on the Deribit perpetual of BTC and of ETH with Deribit's taker fee. The session is one UTC day, and each strategy decides every 30 minutes.

- `VwapMomentum`: intraday momentum in the noise area with the VWAP as a trailing stop (Zarattini, Aziz and Barbon 2024). The candidate.
- `VwapTrend`: long above the session VWAP, short below it, flat at the session end. The baseline.
- `VwapBandFade`: short above the VWAP plus 2 standard deviations, long below the VWAP minus 2, out at the VWAP or at 3. The control.

The notebook runs the train and the validation parts of 2022-01-01 to 2022-07-01 and never the test part. The server needs the candles of the five exchanges for these days, and Coinbase International has none before its listing: the run skips those days and lists them as known gaps.

Set `SBT2_SERVER_TOKEN`, and `SBT2_SERVER_URL` if the server is not on this machine.

In [ ]:
import os

import pandas as pd

from sbt2.lab import Lab
from sbt2.strategies.vwap.band_fade import VwapBandFade
from sbt2.strategies.vwap.momentum import VwapMomentum
from sbt2.strategies.vwap.trend import VwapTrend

url = os.environ.get("SBT2_SERVER_URL", "ws://127.0.0.1:8765")
lab = await Lab.connect(url, os.environ["SBT2_SERVER_TOKEN"])

In [ ]:
STRATEGIES = {
    "momentum": VwapMomentum,
    "trend": VwapTrend,
    "band fade": VwapBandFade,
}


def spec(coin: str) -> dict:
    feeds = [
        f"{coin}USDT-PERP.BINANCE",
        f"{coin}-USDT-SWAP.OKX",
        f"{coin}-PERP.COINBASE_INTX",
        f"{coin}USDT-LINEAR.BYBIT",
        f"{coin}-PERPETUAL.DERIBIT",
    ]
    return {
        "instruments": [f"{coin}-PERPETUAL.DERIBIT"],
        "period": ["2022-01-01", "2022-07-01"],
        "execution": {
            "fee_model": {
                "kind": "maker_taker",
                "config": {"maker_rate": "0.00015", "taker_rate": "0.00035"},
            }
        },
        "capital": f"1 {coin}",
        "bars": "candles",
        "split": {"validation": 0.2, "test": 0.2},
        "part": ["train", "validation"],
        "params": {"feeds": [feeds]},
    }

In [ ]:
runs = {}
for coin in ("BTC", "ETH"):
    for name, strategy in STRATEGIES.items():
        runs[coin, name] = await lab.run(spec(coin), strategy)
pd.concat({key: each.table() for key, each in runs.items()}, names=["coin", "strategy"])

The statistics over the trades of each run.

In [ ]:
statistics = {}
for key, each in runs.items():
    for run in each:
        statistics[*key, run.part] = await run.trade_statistics()
pd.DataFrame(statistics)

The tearsheet of the momentum strategy on BTC, train part.

In [ ]:
await runs["BTC", "momentum"][0].tearsheet()

The tearsheet of the trend strategy on BTC, train part.

In [ ]:
await runs["BTC", "trend"][0].tearsheet()

The tearsheet of the band fade on BTC, train part.

In [ ]:
await runs["BTC", "band fade"][0].tearsheet()

The tearsheet of the momentum strategy on ETH, train part.

In [ ]:
await runs["ETH", "momentum"][0].tearsheet()

## Comparison

On the first run (2022-01-01 to 2022-07-01, train and validation) the costs decide. The trend baseline loses most of the capital on both coins: it flips about 15 times a day, and Deribit's taker fee of 0.035 % a side is more than its edge. The band fade loses less on the train part and as much on the validation part. The momentum strategy trades the least and loses the least on the train part; on the validation part it is flat on BTC (+1.6 %, Sharpe 0.6) and on ETH (-0.6 %). It is the only candidate that does not lose on both parts, and a part this short does not prove an edge.

In [ ]:
await lab.close()